# Import

In [ ]:
import pandas as pd
import scanpy as sc

import sys

sys.path.append("../../")
from curation_tools.curation_tools import CuratedDataset
from curation_tools.perturbseq_anndata_schema import ObsSchema, VarSchema

from curation_tools.unified_metadata_schema.unified_metadata_schema import Experiment

# Pre-process the data

In [ ]:
import GEOparse

data_source_link = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE228nnn/GSE228115/suppl/GSE228115%5FRNA%5Fcounts.txt.gz"

data_df = pd.read_table(
    data_source_link,
    index_col=0,
    header=0,
    compression="gzip",
)
metadata_df = GEOparse.get_GEO(
    'GSE228115', destdir=".", silent=True)

metadata_df = metadata_df.phenotype_data

display(data_df.head())
display(metadata_df.head())

In [ ]:
# assign index to metadata_df
metadata_df.index = metadata_df['title']
# clean up column names
metadata_df.columns = metadata_df.columns.str.replace('_ch1', '')
# drop unnecessary columns
metadata_df = metadata_df[['title', 'organism', 'characteristics.0.cell line', 'characteristics.1.cell type']]

display(metadata_df.head())

In [ ]:
# check if sample names match
assert set(data_df.columns) == set(metadata_df['title'])
assert len(data_df.columns) == len(metadata_df)

In [ ]:
# convert to AnnData
adata = sc.AnnData(
    X=data_df.T, obs=metadata_df, var=data_df.index.to_frame().rename(columns={0: 'gene_symbol'})
)

In [ ]:
# save the non-curated AnnData object
adata.write_h5ad("../non_curated/h5ad/rogers_2024.h5ad")

# Initialise the dataset object

In [ ]:
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path = "../non_curated/h5ad/rogers_2024.h5ad"
)

# Load the dataset

In [ ]:
# cur_data.download_data()
cur_data.load_data()
# show the data
cur_data.show_obs()

In [ ]:
cur_data.show_var()

# OBS slot curation

#### Add `perturbation_name` column based on the `title`

In [ ]:
cur_data.create_columns(
    slot = "obs",
    col_dict={
        "perturbation_name": cur_data.adata.obs['title']
    },
    overwrite=True
)

cur_data.replace_entries(
    slot="obs",
    column="perturbation_name",
    map_dict={'CRISPRi_target[_|-]':''}
)

#### Add `perturbed_target_coord` column based on the `perturbation_name`

In [ ]:
cur_data.create_columns(
    slot = "obs",
    col_dict={
        "perturbed_target_coord": cur_data.adata.obs['perturbation_name']
    },
    overwrite=True
)

#### Clean up `perturbation_target_symbol` column

In [ ]:
cur_data.replace_entries(
    slot="obs",
    column="perturbed_target_coord",
    map_dict={
        '_rep\d{1,}': '',
        '-': ':',
        'Safe Harbor': 'control_nontargeting',
        'GFP': 'control_nontargeting',
        'KOLF2.1J_NGN2': 'control_untreated',
        'iCellGluta': 'control_untreated'
    }
)

### Show unique perturbations

In [ ]:
cur_data.show_unique(slot = 'obs', column = 'perturbed_target_coord')

### Add `perturbed_target_number` column

In [ ]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_coord',
    count_column_name='perturbed_target_number',
    sep='|'
)

### Add `perturbed_target_biotype`

In [ ]:
cur_data.create_columns(
    slot='obs',
    col_dict={
        'perturbed_target_biotype': ['regulatory' if e.startswith('chr') else None for e in cur_data.adata.obs['perturbed_target_coord']]
    },
    overwrite=True
)     

### Add empty `perturbed_target_ensg`, `perturbed_target_symbol`, `perturbed_target_biotype` columns

In [ ]:
cur_data.create_columns(
    slot='obs',
    col_dict={
        'perturbed_target_ensg': None, 
        'perturbed_target_symbol': None
    }
)

In [ ]:
cur_data.show_obs(['perturbation_name', 'perturbed_target_coord', 'perturbed_target_number'])

### Add treatment information

Add treatment information with the dataset

In [ ]:
cur_data.create_columns(
    slot="obs",
    col_dict={
        "treatment_label": None, 
        "treatment_id": None
    }
)

### Add perturbation information

In [ ]:
cur_data.create_columns(
    slot="obs",
    col_dict={
        "perturbation_type_label": "CRISPRi", 
        "perturbation_type_id": None
    }
)

### Add timepoint information

In [ ]:
cur_data.create_columns(
    slot="obs",
    col_dict={"timepoint": "P0DT0H0M0S"},
)

### Add model system information

In [ ]:
cur_data.create_columns(
    slot="obs",
    col_dict={
        "model_system_label": "stem cell-derived cells", 
        "model_system_id": None
    }
)

### Add tissue information

In [ ]:
cur_data.create_columns(
    slot='obs',
    col_dict={
        'tissue': 'brain'
    }
)

cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

### Add cell type information

In [ ]:
cur_data.show_unique(
    slot='obs',
    column='characteristics.1.cell type'
)

In [ ]:
# KOLF2.1J NGN2 are incorrectly labeled as neurons, but they are actually glutamatergic neurons
# so we will map them to the correct cell type
cur_data.map_values_from_column(
    ref_col='characteristics.0.cell line',
    target_col='characteristics.1.cell type',
    map_dict={
        'KOLF2.1J NGN2': 'glutamatergic neurons'
    }
)

In [ ]:
cur_data.replace_entries(
    slot='obs',
    column='characteristics.1.cell type',
    map_dict={
        r'Gluta Neurons': 'glutamatergic neurons'
    }
)

cur_data.standardize_ontology(
    input_column='characteristics.1.cell type',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

### Add cell line information

In [ ]:
# correct cell line name from cellosaurus (https://www.cellosaurus.org/CVCL_D1J6)
# correct the Fuji cell line name (https://www.fujifilmcdi.com/icell-glutaneurons-01279-ggln01279)

cur_data.replace_entries(
    slot='obs',
    column='characteristics.0.cell line',
    map_dict={
        r'KOLF2.1J NGN2': 'KOLF2.1J AAVS1-TREG3-NGN2',
        r'iCellGluta': 'iCell GlutaNeurons',
    }
)

In [ ]:
# standardization won't work because these cells are not in the ontology
cur_data.standardize_ontology(
    input_column='characteristics.0.cell line',
    column_type='term_name',
    ontology_type='cell_line'
)

# proceed with manually creating the columns
cur_data.create_columns(
    slot='obs',
    col_dict={
        "cell_line_label": cur_data.adata.obs['characteristics.0.cell line'],
        "cell_line_id": None
    }
)

### Add disease information

In [ ]:
cur_data.create_columns(
    slot='obs',
    col_dict={
        "disease_label": None,
        "disease_id": None
    }
)

### Add species information

In [ ]:
cur_data.create_columns(
    slot="obs",
    col_dict={
        "species": "Homo sapiens"
    }
)

### Add sex information

In [ ]:
cur_data.create_columns(
    slot="obs",
    col_dict={
        "sex_label": None, 
        "sex_id": None
    }
)

cur_data.map_values_from_column(
    ref_col='cell_line_label',
    target_col='sex_label',
    map_dict={
        'BC1': 'female', # https://www.cellosaurus.org/CVCL_RX99
        'XCL4': 'female', # https://cdn.stemcell.com/media/files/pis/DX21378-PIS_1_1_0.pdf
        'iCell GlutaNeurons': 'male', # https://www.fujifilmcdi.com/icell-glutaneurons-01279-ggln01279
        'KOLF2.1J AAVS1-TREG3-NGN2': 'male' # https://www.cellosaurus.org/CVCL_D1J6
    }
)

### Add developmental stage information

In [ ]:
cur_data.create_columns(
    slot="obs",
    col_dict={
        "developmental_stage_label": "adult", 
        "developmental_stage_id": None
    }
)

### Add guide RNA sequences

In [ ]:
cur_data.create_columns(
    slot='obs',
    col_dict={
        'guide_sequence': None
    }
)

### Match schema column order

In [ ]:
cur_data.match_schema_columns(slot='obs')

### Validate obs metadata

In [ ]:
cur_data.validate_data(slot='obs')

# VAR slot curation

### Standardise genes

In [ ]:
cur_data.show_var()

In [ ]:
cur_data.standardize_genes(
    slot="var", input_column="gene_symbol", input_column_type="gene_symbol"
)

### Validate var metadata

In [ ]:
cur_data.validate_data(slot='var')

# Metadata curation

### Auto-populate available metadata

In [ ]:
cur_data.populate_exp_metadata()

### Manually curate metadata

Study details

In [ ]:
cur_data.add_exp_metadata(
    metadata_slot="study",
    metadata={
        "title": "Neuronal MAPT expression is mediated by long-range interactions with cis-regulatory elements",
        "study_uri": "https://doi.org/10.1016/j.ajhg.2023.12.015",
        "year": 2024,
        "first_author": {"first_name": "Brianne", "last_name": "Rogers"},
        "last_author": {"first_name": "Jesse", "last_name": "Cochran"},
    }
)

Experiment details

In [ ]:
cur_data.exp_metadata

In [ ]:
cur_data.add_exp_metadata(
    metadata_slot='experiment',
    metadata={
        "title": "CRISPRi to profile potential cis-regulatory elements of MAPT",
        "summary": "A total of 384 samples, including non-targeting controls (AAVS1 GSH and GFP) and untrgeted controls (KOLF2.1J NGN2 and iCell GlutaNeurons), were perturbed using dCas-KRAB and sgRNAs targeting potential cis-regulatory elements of MAPT. Perturbations were performed on two different neuronal progenitor cell lines (XLC4, BC1) differentiated into neurons.",
        "replicates": "Multiple replicates",
        "number_of_samples": 384
    }
)

Perturbation details

In [ ]:
cur_data.add_exp_metadata(
    metadata_slot='perturbation',
    metadata={
         "library_generation_type": {
            "term_id": "EFO:0022868",
            "term_label": "endogenous",
        },
        "library_generation_method": {
            "term_id": "EFO:0022895",
            "term_label": "dCas9-KRAB",
        },
        "enzyme_delivery_method": {
            "term_id": None,
            "term_label": "lentiviral transduction",
        },
        "library_delivery_method": {
            "term_id": None,
            "term_label": "lentiviral transduction",
        },
        "enzyme_integration_state": {
            "term_id": None,
            "term_label": "random locus integration",
        },
        "library_integration_state": {
            "term_id": None,
            "term_label": "random locus integration",
        },
        "enzyme_expression_control": {
            "term_id": None,
            "term_label": "constitutive expression",
        },
        "library_expression_control": {
            "term_id": None,
            "term_label": "constitutive expression",
        },
        "library": {
            "library_name": "custom",
            "accession": None,
            "library_format": {
                "term_id": None,
                "term_label": "arrayed",
            },
            "library_scope": {
                "term_id": None,
                "term_label": "focused",
            },
            "library_perturbation_type": [
                {
                    "term_id": None,
                    "term_label": "inhibition",
                },
            ],
            "manufacturer": "Cochran",
            "lentiviral_generation": "3",
            "grnas_per_target": "1+",
            "total_grnas": "96",
            "total_variants": None
        }
    }
)

Assay details

In [ ]:
cur_data.add_exp_metadata(
    metadata_slot='assay',
    metadata={
        "readout_dimensionality": {
            "term_id": None,
            "term_label": "high-dimensional assay",
        },
        "readout_type": {
            "term_id": None,
            "term_label": "transcriptomic",
        },
        "readout_technology": {
            "term_id": None,
            "term_label": "rna-seq",
        },
        "method_name": {
            "term_id": None,
            "term_label": "CRISPR screen",
        },
        "method_uri": None,
        "sequencing_library_kit": {
            "term_id": None,
            "term_label": "QuantSeq 3′ mRNA-Seq Library Prep Kit FWD",
        },
        "sequencing_platform": {"term_id": None, "term_label": "Illumina NovaSeq 6000"},
        "sequencing_strategy": {"term_id": None, "term_label": "barcode sequencing"},
        "software_counts": {"term_id": None, "term_label": "htseq-count"},
        "software_analysis": {"term_id": None, "term_label": "custom"},
        "reference_genome": {
            "term_id": None,
            "term_label": "GRCh38",
        }
    }
)

Model system details

In [ ]:
cur_data.add_exp_metadata(
    metadata_slot='model_system',
    metadata={
        "species": "Homo sapiens",
        "passage_number": None,
        }
)

Associated dataset details

In [ ]:
cur_data.add_exp_metadata(
    metadata_slot='associated_datasets',
    metadata=[
        {
            "dataset_accession": "GSE228115",
            "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE228115",
            "dataset_description": "Raw counts",
            "dataset_file_name": "GSE228115_RNA_counts.txt.gz",
        }
    ]
)

### Validate metadata

In [ ]:
cur_data.validate_exp_metadata()

# Save the dataset

In [ ]:
cur_data.save_curated_data()